# Song Recommendation System with Million Song Dataset

**Problem Statement:** Recommender systems predict a user's preference for songs they haven't listened to yet. In this project, we build a song recommendation engine using the Kaggle Million Song dataset, exploring Cosine Similarity and Content-Based Filtering (using audio features like danceability, energy, tempo, and valence.)

**Useful Links As We Get Started:** \
*Kaggle dataset* https://www.kaggle.com/datasets/solomonameh/spotify-music-dataset \
*Pandas intro:* https://pandas.pydata.org/docs/getting_started/index.html#getting-started \
*Cosine similarity:* https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html

In [6]:
# imports
# %pip install pandas numpy scikit-learn

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

# Merge the two datasets

**Function 1:** Merge high_popularity_spotify_data.csv and low_popularity_spotify_data.csv into one dataset

In [7]:
def load_spotify_dataset(high_pop, low_pop):
    # read the high popularity CSV into a DataFrame
    high_df = pd.read_csv(high_pop)
    # read the low popularity CSV into a DataFrame
    low_df  = pd.read_csv(low_pop)
    # combine both DataFrames into one, resetting the index (so we don't have index duplicates among songs)
    df = pd.concat([high_df, low_df], ignore_index=True)
    # return the merged DataFrame!
    return df

# Preprocess the data

**Function 2:** Normalize audio features and clean up the data

In [8]:
def preprocess_spotify_dataset(df):
    # remove duplicates in case (each song has a unique track_id assigned by Spotify, remove if there
    # are two of the same track_ids
    df = df.drop_duplicates(subset='track_id')
    # remove any rows that have missing values (nulls) in any column
    df = df.dropna()

    # a list of the Spotify audio features we want to use for modeling
    audio_features = [
        'danceability', 'energy', 'loudness', 'speechiness',
        'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
    ]
    # keep only the features that actually exist as columns in our dataset
    audio_features = [f for f in audio_features if f in df.columns]

    # create a scaler that will compress all values into the range [0, 1] aka we are normalizing our features!
    scaler = MinMaxScaler()
    # fit scaler (compressing our data to the range [0, 1] to our data and apply it in place
    df[audio_features] = scaler.fit_transform(df[audio_features])

    # print rows x columns so we can see how much data remains
    print(f"Dataset shape after cleaning: {df.shape}")
    # return the cleaned up and normalized dataframe

    # prints audio features first 10 rows
    print(df[audio_features].head(10).to_string())

    # prints full 29 columns for first 3 rows
    print(pd.DataFrame(df).head(3).to_string())
    
    return df

# Train a very basic recommender

**Function 3:** Builds a cosine similarity matrix over audio features, where each row is a song, each column is a song and the value at [row, column] is how similar those songs are (0 to 1). Returns the similarity matrix and the track index for song lookups.

In [9]:
def train_song_recommender(df):

    # same list as preprocessing data function for audio features
    audio_features = [
        'danceability', 'energy', 'loudness', 'speechiness',
        'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
    ]

    # same check as above since we want to make sure we only keep features present in the dataframe
    audio_features = [f for f in audio_features if f in df.columns]

    # extract just the numeric feature values as a NumPy array
    feature_matrix = df[audio_features].values
    # compute how similar every song is to every other song; scores range from 0 (no similarity) to 1 (identical)
    similarity_matrix = cosine_similarity(feature_matrix)

    # should be (n_songs x n_songs)
    print(f"Similarity matrix shape: {similarity_matrix.shape}")
    # return the matrix and track IDs so we can map scores back to songs
    
    return similarity_matrix, df['track_id'].values

**Run our functions:** A tuple of the dataset (unique songs, 29 for number of columns), where columns contains the Spotify tags of 'energy', 'tempo', 'danceability', 'playlist_genre', 'loudness', 'liveness', 'valence', 'track_artist', 'time_signature', 'speechiness', 'track_popularity', 'track_href', 'uri', 'track_album_name', 'playlist_name', 'analysis_url', 'track_id', 'track_name', 'track_album_release_date', 'instrumentalness', 'track_album_id', 'mode', 'key', 'duration_ms', 'acousticness', 'id', 'playlist_subgenre', 'type', 'playlist_id'.


In [10]:
# run our functions
df = load_spotify_dataset("high_popularity_spotify_data.csv",
                          "low_popularity_spotify_data.csv")
df = preprocess_spotify_dataset(df)
similarity_matrix, track_index = train_song_recommender(df)

Dataset shape after cleaning: (4493, 29)
   danceability    energy  loudness  speechiness  acousticness  instrumentalness  liveness   valence     tempo
0      0.502228  0.593104  0.815842     0.009391      0.309234          0.000000  0.105428  0.527888  0.568015
1      0.747853  0.507916  0.767368     0.015357      0.200800          0.061352  0.100209  0.426572  0.293725
2      0.538094  0.809581  0.888898     0.016462      0.214857          0.000000  0.144050  0.357635  0.312204
3      0.664167  0.911806  0.890902     0.045851      0.094274          0.000000  0.295407  0.790056  0.335073
4      0.780459  0.784526  0.882661     0.263065      0.028410          0.000000  0.348643  0.949864  0.521729
5      0.696772  0.583082  0.852633     0.015136      0.050398          0.000000  0.070042  0.789012  0.354462
6      0.663080  0.562036  0.840930     0.021213      0.496986          0.009707  0.077662  0.847504  0.527123
7      0.443539  0.247343  0.730293     0.023423      0.614456         

/Users/alisabui/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alisabui/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alisabui/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
